In [1]:
import sys
!{sys.executable} -m pip install nba_api

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
from nba_api.stats.endpoints import leaguedashteamstats

df = leaguedashteamstats.LeagueDashTeamStats(
    season="2010-11",
    season_type_all_star="Regular Season",
).get_data_frames()[0]

df["EFG_PCT"] = (df["FGM"] + 0.5 * df["FG3M"]) / df["FGA"]

print(df[["TEAM_NAME", "EFG_PCT", "W_PCT", "PTS"]].head())

             TEAM_NAME   EFG_PCT  W_PCT   PTS
0        Atlanta Hawks  0.501167  0.537  7790
1       Boston Celtics  0.518894  0.683  7913
2    Charlotte Bobcats  0.482247  0.415  7650
3        Chicago Bulls  0.500607  0.756  8087
4  Cleveland Cavaliers  0.472469  0.232  7827


In [3]:
print(df.columns.tolist())

['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'EFG_PCT']


In [16]:
from nba_api.stats.endpoints import leaguedashteamstats

adv = leaguedashteamstats.LeagueDashTeamStats(
    season="2010-11",
    season_type_all_star="Regular Season",
    measure_type_detailed_defense="Advanced"
).get_data_frames()[0]

print([c for c in adv.columns if "RTG" in c or "RATING" in c])
print(adv[["TEAM_NAME", "OFF_RATING", "DEF_RATING", "NET_RATING"]].head())

base = df[["TEAM_ID","TEAM_NAME","W_PCT","FGM","FGA","FG3M"]].copy()
base["EFG_PCT"] = (base["FGM"] + 0.5 * base["FG3M"]) / base["FGA"]

adv_small = adv[["TEAM_ID","OFF_RATING"]].copy() if "OFF_RATING" in adv.columns else adv[["TEAM_ID","OFF_RTG"]].copy()

out = base.merge(adv_small, on="TEAM_ID", how="left")
print(out[["TEAM_NAME","EFG_PCT","W_PCT"] + list(adv_small.columns.drop("TEAM_ID"))].head())

['E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK']
             TEAM_NAME  OFF_RATING  DEF_RATING  NET_RATING
0        Atlanta Hawks       105.1       105.8        -0.7
1       Boston Celtics       105.5        99.8         5.7
2    Charlotte Bobcats       102.8       107.0        -4.2
3        Chicago Bulls       107.2        99.5         7.8
4  Cleveland Cavaliers       101.5       110.9        -9.5
             TEAM_NAME   EFG_PCT  W_PCT  OFF_RATING
0        Atlanta Hawks  0.501167  0.537       105.1
1       Boston Celtics  0.518894  0.683       105.5
2    Charlotte Bobcats  0.482247  0.415       102.8
3        Chicago Bulls  0.500607  0.756       107.2
4  Cleveland Cavaliers  0.472469  0.232       101.5


In [52]:
from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import teams
import pandas as pd

team_id = [t["id"] for t in teams.get_teams() if t["abbreviation"] == "BOS"][0]

shots = shotchartdetail.ShotChartDetail(
    team_id=team_id,
    player_id=0,
    season_nullable="2014-15",
    season_type_all_star="Regular Season",
    context_measure_simple="FGA"
).get_data_frames()[0]

print(shots.columns.tolist())
print(shots[["TEAM_NAME","PLAYER_NAME","SHOT_DISTANCE","SHOT_MADE_FLAG","SHOT_TYPE","LOC_X","LOC_Y"]].head(10))

shots[["PLAYER_NAME","TEAM_NAME","SHOT_DISTANCE","SHOT_MADE_FLAG","SHOT_TYPE", "SHOT_ZONE_BASIC", "SHOT_ZONE_AREA", "SHOT_ZONE_RANGE"]].head(20)

shots.to_csv("celtics_2024_25_shots.csv", index=False)

shots["SHOT_ZONE_BASIC"].value_counts()

['GRID_TYPE', 'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME', 'PERIOD', 'MINUTES_REMAINING', 'SECONDS_REMAINING', 'EVENT_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_ZONE_RANGE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y', 'SHOT_ATTEMPTED_FLAG', 'SHOT_MADE_FLAG', 'GAME_DATE', 'HTM', 'VTM']
        TEAM_NAME      PLAYER_NAME  SHOT_DISTANCE  SHOT_MADE_FLAG  \
0  Boston Celtics     Kelly Olynyk           24.0               0   
1  Boston Celtics       Jeff Green            1.0               1   
2  Boston Celtics    Avery Bradley           25.0               1   
3  Boston Celtics    Avery Bradley           25.0               0   
4  Boston Celtics  Jared Sullinger            4.0               0   
5  Boston Celtics  Jared Sullinger            1.0               1   
6  Boston Celtics     Kelly Olynyk            0.0               0   
7  Boston Celtics       Jeff Green           25.0               1   
8  Boston Celtics    Avery Bradl

SHOT_ZONE_BASIC
Restricted Area          2233
Mid-Range                1995
Above the Break 3        1573
In The Paint (Non-RA)     962
Left Corner 3             229
Right Corner 3            201
Backcourt                  16
Name: count, dtype: int64

In [53]:
import numpy as np
import pandas as pd

shots = shots.copy()
shots = shots[shots["SHOT_ZONE_BASIC"] != "Backcourt"].copy()

is_three = shots["SHOT_TYPE"].str.contains("3PT", na=False)

shots["zone"] = np.select(
    [
        shots["SHOT_ZONE_BASIC"].isin(["Left Corner 3", "Right Corner 3"]),
        shots["SHOT_ZONE_BASIC"].eq("Above the Break 3"),
        (~is_three) & (shots["SHOT_DISTANCE"] <= 3),
        (~is_three) & (shots["SHOT_DISTANCE"] > 3) & (shots["SHOT_DISTANCE"] <= 10),
        (~is_three) & (shots["SHOT_DISTANCE"] > 10),
    ],
    [
        "Corner 3",
        "Above-break 3",
        "Rim (0–3)",
        "Short mid (3–10)",
        "Long mid (10–22)",
    ],
    default=pd.NA
)

shots = shots.dropna(subset=["zone"]).copy()

shots["zone"].value_counts()

zone
Rim (0–3)           2233
Long mid (10–22)    2057
Above-break 3       1573
Short mid (3–10)     900
Corner 3             430
Name: count, dtype: int64

In [54]:
league_season = (
    shots.groupby("zone")
         .agg(
             FGA=("SHOT_ATTEMPTED_FLAG", "sum"),
             FGM=("SHOT_MADE_FLAG", "sum")
         )
         .reset_index()
)

league_season["FG_PCT"] = league_season["FGM"] / league_season["FGA"]
league_season["FREQ"] = league_season["FGA"] / league_season["FGA"].sum()
league_season["SEASON"] = "2014-15"

league_season

,zone,FGA,FGM,FG_PCT,FREQ,SEASON
0,Above-break 3,1573,500,0.317864,0.218685,2014-15
1,Corner 3,430,160,0.372093,0.059780,2014-15
2,Long mid (10–22),2057,857,0.416626,0.285972,2014-15
3,Rim (0–3),2233,1336,0.598298,0.310441,2014-15
4,Short mid (3–10),900,340,0.377778,0.125122,2014-15


In [66]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import shotchartdetail
import pandas as pd
import time

team_list = teams.get_teams()
team_ids = [t["id"] for t in team_list]

all_shots = []

for tid in team_ids:
    print("Pulling team:", tid)
    
    df = shotchartdetail.ShotChartDetail(
        team_id=tid,
        player_id=0,
        season_nullable="2010-11",
        season_type_all_star="Regular Season",
        context_measure_simple="FGA"
    ).get_data_frames()[0]
    
    all_shots.append(df)
    time.sleep(0.7)  # prevent rate limit

league_shots = pd.concat(all_shots, ignore_index=True)

print("Total shots pulled:", len(league_shots))

Pulling team: 1610612737
Pulling team: 1610612738
Pulling team: 1610612739
Pulling team: 1610612740
Pulling team: 1610612741
Pulling team: 1610612742
Pulling team: 1610612743
Pulling team: 1610612744
Pulling team: 1610612745
Pulling team: 1610612746
Pulling team: 1610612747
Pulling team: 1610612748
Pulling team: 1610612749
Pulling team: 1610612750
Pulling team: 1610612751
Pulling team: 1610612752
Pulling team: 1610612753
Pulling team: 1610612754
Pulling team: 1610612755
Pulling team: 1610612756
Pulling team: 1610612757
Pulling team: 1610612758
Pulling team: 1610612759
Pulling team: 1610612760
Pulling team: 1610612761
Pulling team: 1610612762
Pulling team: 1610612763
Pulling team: 1610612764
Pulling team: 1610612765
Pulling team: 1610612766
Total shots pulled: 199790


In [67]:
import numpy as np
import pandas as pd

league_shots = league_shots.copy()

# drop backcourt (not part of normal shot selection)
league_shots = league_shots[league_shots["SHOT_ZONE_BASIC"] != "Backcourt"].copy()

is_three = league_shots["SHOT_TYPE"].str.contains("3PT", na=False)

league_shots["zone"] = np.select(
    [
        league_shots["SHOT_ZONE_BASIC"].isin(["Left Corner 3", "Right Corner 3"]),
        league_shots["SHOT_ZONE_BASIC"].eq("Above the Break 3"),
        (~is_three) & (league_shots["SHOT_DISTANCE"] <= 3),
        (~is_three) & (league_shots["SHOT_DISTANCE"] > 3) & (league_shots["SHOT_DISTANCE"] <= 10),
        (~is_three) & (league_shots["SHOT_DISTANCE"] > 10),
    ],
    [
        "Corner 3",
        "Above-break 3",
        "Rim (0–3)",
        "Short mid (3–10)",
        "Long mid (10–22)",
    ],
    default=pd.NA
)

# remove anything that still didn't map (rare)
league_shots = league_shots.dropna(subset=["zone"]).copy()

league_season = (
    league_shots.groupby("zone")
         .agg(
             FGA=("SHOT_ATTEMPTED_FLAG", "sum"),
             FGM=("SHOT_MADE_FLAG", "sum")
         )
         .reset_index()
)

league_season["FG_PCT"] = league_season["FGM"] / league_season["FGA"]
league_season["FREQ"] = league_season["FGA"] / league_season["FGA"].sum()
league_season["SEASON"] = "2010-11"

league_season.sort_values("FGA", ascending=False)

,zone,FGA,FGM,FG_PCT,FREQ,SEASON
3,Rim (0–3),62928,39009,0.619899,0.315758,2010-11
2,Long mid (10–22),61638,24837,0.402949,0.309285,2010-11
0,Above-break 3,31486,10999,0.349330,0.157989,2010-11
4,Short mid (3–10),30895,11887,0.384755,0.155024,2010-11
1,Corner 3,12345,4867,0.394249,0.061944,2010-11


In [68]:
import time
import numpy as np
import pandas as pd
from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import teams

team_ids = [t["id"] for t in teams.get_teams()]
seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2010, 2025)]

league_rows = []

for season in seasons:
    print("Season:", season)

    all_shots = []
    for tid in team_ids:
        df = shotchartdetail.ShotChartDetail(
            team_id=tid,
            player_id=0,
            season_nullable=season,
            season_type_all_star="Regular Season",
            context_measure_simple="FGA"
        ).get_data_frames()[0]

        all_shots.append(df)
        time.sleep(0.7)  # rate-limit friendly

    league_shots = pd.concat(all_shots, ignore_index=True)

    # drop backcourt (not normal shot selection)
    league_shots = league_shots[league_shots["SHOT_ZONE_BASIC"] != "Backcourt"].copy()

    is_three = league_shots["SHOT_TYPE"].str.contains("3PT", na=False)

    league_shots["zone"] = np.select(
        [
            league_shots["SHOT_ZONE_BASIC"].isin(["Left Corner 3", "Right Corner 3"]),
            league_shots["SHOT_ZONE_BASIC"].eq("Above the Break 3"),
            (~is_three) & (league_shots["SHOT_DISTANCE"] <= 3),
            (~is_three) & (league_shots["SHOT_DISTANCE"] > 3) & (league_shots["SHOT_DISTANCE"] <= 10),
            (~is_three) & (league_shots["SHOT_DISTANCE"] > 10),
        ],
        ["Corner 3", "Above-break 3", "Rim (0–3)", "Short mid (3–10)", "Long mid (10–22)"],
        default=pd.NA
    )

    league_shots = league_shots.dropna(subset=["zone"]).copy()

    league_season = (
        league_shots.groupby("zone")
        .agg(FGA=("SHOT_ATTEMPTED_FLAG", "sum"),
             FGM=("SHOT_MADE_FLAG", "sum"))
        .reset_index()
    )

    league_season["FG_PCT"] = league_season["FGM"] / league_season["FGA"]
    league_season["FREQ"]   = league_season["FGA"] / league_season["FGA"].sum()
    league_season["SEASON"] = season

    league_rows.append(league_season)

league_all = pd.concat(league_rows, ignore_index=True)

league_all.to_csv("nba_league_zone_summary_2010_11_to_2024_25.csv", index=False)
print("Saved:", "nba_league_zone_summary_2010_11_to_2024_25.csv")

Season: 2010-11
Season: 2011-12
Season: 2012-13
Season: 2013-14
Season: 2014-15
Season: 2015-16
Season: 2016-17
Season: 2017-18
Season: 2018-19
Season: 2019-20
Season: 2020-21
Season: 2021-22
Season: 2022-23
Season: 2023-24
Season: 2024-25
Saved: nba_league_zone_summary_2010_11_to_2024_25.csv


In [4]:
from nba_team_shot_trends import save_dataset

result = save_dataset(pause_seconds=0.7)

team_shots_long = result.shots_long
team_shots_wide = result.shots_wide
team_records = result.team_records
team_shot_trends = result.merged

team_shot_trends.head()


Pulling shot chart data for 2010-11
Pulling shot chart data for 2011-12
Pulling shot chart data for 2012-13
Pulling shot chart data for 2013-14


KeyboardInterrupt: 

In [5]:
import time
import numpy as np
import pandas as pd
from nba_api.stats.endpoints import shotchartdetail, leaguedashteamstats
from nba_api.stats.static import teams

# ── Config ────────────────────────────────────────────────────────────────────
SEASONS = [f"{y}-{str(y+1)[-2:]}" for y in range(2010, 2025)]
ZONE_CONDITIONS = lambda df: [
    df["SHOT_ZONE_BASIC"].isin(["Left Corner 3", "Right Corner 3"]),
    df["SHOT_ZONE_BASIC"].eq("Above the Break 3"),
    (~df["SHOT_TYPE"].str.contains("3PT", na=False)) & (df["SHOT_DISTANCE"] <= 3),
    (~df["SHOT_TYPE"].str.contains("3PT", na=False)) & (df["SHOT_DISTANCE"] > 3)  & (df["SHOT_DISTANCE"] <= 10),
    (~df["SHOT_TYPE"].str.contains("3PT", na=False)) & (df["SHOT_DISTANCE"] > 10),
]
ZONE_LABELS = ["Corner 3", "Above-break 3", "Rim (0–3)", "Short mid (3–10)", "Long mid (10–22)"]

team_list = teams.get_teams()

all_rows = []

for season in SEASONS:
    print(f"\n── Season: {season} ──")

    # ── 1. Fetch win % for every team this season ──────────────────────────────
    wstats = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense="Base",
    ).get_data_frames()[0][["TEAM_ID", "TEAM_NAME", "W_PCT", "GP"]]
    time.sleep(0.6)

    # ── 2. Fetch shots per team ────────────────────────────────────────────────
    for team in team_list:
        tid  = team["id"]
        tname = team["full_name"]
        print(f"  {tname}", end=" ", flush=True)

        try:
            shot_df = shotchartdetail.ShotChartDetail(
                team_id=tid,
                player_id=0,
                season_nullable=season,
                season_type_all_star="Regular Season",
                context_measure_simple="FGA",
            ).get_data_frames()[0]
            time.sleep(0.7)
        except Exception as e:
            print(f"[ERROR: {e}]")
            continue

        if shot_df.empty:
            print("[no data]")
            continue

        # ── 3. Zone classification ─────────────────────────────────────────────
        shot_df = shot_df[shot_df["SHOT_ZONE_BASIC"] != "Backcourt"].copy()
        shot_df["zone"] = np.select(ZONE_CONDITIONS(shot_df), ZONE_LABELS, default=pd.NA)
        shot_df = shot_df.dropna(subset=["zone"])

        # ── 4. Aggregate to zone-level stats ───────────────────────────────────
        team_zone = (
            shot_df.groupby("zone")
            .agg(FGA=("SHOT_ATTEMPTED_FLAG", "sum"),
                 FGM=("SHOT_MADE_FLAG",      "sum"))
            .reset_index()
        )
        total_fga = team_zone["FGA"].sum()
        team_zone["FG_PCT"] = team_zone["FGM"] / team_zone["FGA"]
        team_zone["FREQ"]   = team_zone["FGA"] / total_fga
        team_zone["PPS"]    = (team_zone["FGM"] / team_zone["FGA"]) * np.where(
            team_zone["zone"].str.contains("3"), 3, 2
        )   # points per shot by zone

        # ── 5. Tag team & season ───────────────────────────────────────────────
        team_zone["TEAM_ID"]   = tid
        team_zone["TEAM_NAME"] = tname
        team_zone["SEASON"]    = season

        all_rows.append(team_zone)
        print("✓")

    print(f"  Win % data: {len(wstats)} teams")

    # Store w_pct separately keyed by (TEAM_ID, SEASON) for the final merge
    wstats["SEASON"] = season
    all_rows_wpct = wstats if season == SEASONS[0] else pd.concat([all_rows_wpct, wstats], ignore_index=True)

# ── 6. Combine & merge win % ──────────────────────────────────────────────────
shots_all = pd.concat(all_rows, ignore_index=True)

final = shots_all.merge(
    all_rows_wpct[["TEAM_ID", "SEASON", "W_PCT", "GP"]],
    on=["TEAM_ID", "SEASON"],
    how="left"
)

# ── 7. Save ───────────────────────────────────────────────────────────────────
OUT = "nba_team_zone_wpct_2010_2025.csv"
final.to_csv(OUT, index=False)
print(f"\nSaved {len(final):,} rows → {OUT}")
print(final[["TEAM_NAME","SEASON","zone","FREQ","FG_PCT","PPS","W_PCT"]].head(10).to_string(index=False))


── Season: 2010-11 ──
  Atlanta Hawks ✓
  Boston Celtics ✓
  Cleveland Cavaliers ✓
  New Orleans Pelicans ✓
  Chicago Bulls ✓
  Dallas Mavericks ✓
  Denver Nuggets ✓
  Golden State Warriors ✓
  Houston Rockets ✓
  Los Angeles Clippers ✓
  Los Angeles Lakers ✓
  Miami Heat ✓
  Milwaukee Bucks ✓
  Minnesota Timberwolves ✓
  Brooklyn Nets ✓
  New York Knicks ✓
  Orlando Magic ✓
  Indiana Pacers ✓
  Philadelphia 76ers ✓
  Phoenix Suns ✓
  Portland Trail Blazers ✓
  Sacramento Kings ✓
  San Antonio Spurs ✓
  Oklahoma City Thunder ✓
  Toronto Raptors ✓
  Utah Jazz ✓
  Memphis Grizzlies ✓
  Washington Wizards ✓
  Detroit Pistons ✓
  Charlotte Hornets ✓
  Win % data: 30 teams

── Season: 2011-12 ──
  Atlanta Hawks ✓
  Boston Celtics ✓
  Cleveland Cavaliers ✓
  New Orleans Pelicans ✓
  Chicago Bulls ✓
  Dallas Mavericks ✓
  Denver Nuggets ✓
  Golden State Warriors ✓
  Houston Rockets ✓
  Los Angeles Clippers ✓
  Los Angeles Lakers ✓
  Miami Heat ✓
  Milwaukee Bucks ✓
  Minnesota Timberwolves ✓